In [1]:
from langchain.chat_models import init_chat_model
from langgraph.func import entrypoint
from langgraph.prebuilt import create_react_agent
from langgraph.store.memory import InMemoryStore

from langmem import create_manage_memory_tool, create_search_memory_tool

model = "anthropic:claude-sonnet-4-6"

# ── STORE SETUP ────────────────────────────────────────────────────────────────
# Shared store — both my_llm (read) and manager agent (write) use the same instance.
# Using OpenAI text-embedding-3-small (1536 dims) for semantic search.
store = InMemoryStore(
    index={
        "dims": 1536,
        "embed": "openai:text-embedding-3-small",
    }
)

# User-facing LLM — Claude Sonnet 4.6 handles the actual conversation.
my_llm = init_chat_model("anthropic:claude-sonnet-4-6")


# ── MEMORY MANAGER PROMPT ──────────────────────────────────────────────────────
# Injected into the manager agent before each reasoning step.
# Searches the store for relevant existing memories so the agent can decide
# whether to insert new ones, update stale ones, or delete contradicted ones.
def manager_prompt(state):
    memories = store.search(
        ("memories",),
        query=state["messages"][-1].content,
    )
    system_msg = f"""You are a memory manager. Extract and manage all important knowledge, rules, and events using the provided tools.

Existing memories:
<memories>
{memories}
</memories>

Use the manage_memory tool to update and contextualize existing memories, create new ones, or delete old ones that are no longer valid.
You can also expand your search of existing memories to augment using the search tool."""
    return [{"role": "system", "content": system_msg}, *state["messages"]]


# ── MEMORY MANAGER AGENT ───────────────────────────────────────────────────────
# Separate ReAct agent dedicated to reading/writing the store.
# The user never interacts with this agent directly.
manager = create_react_agent(
    model=model,
    prompt=manager_prompt,
    tools=[
        create_manage_memory_tool(namespace=("memories",)),
        create_search_memory_tool(namespace=("memories",)),
    ],
)


# ── USER-FACING APP ────────────────────────────────────────────────────────────
@entrypoint(store=store)
def app(messages: list):
    # Step 1: Search the store for memories relevant to the latest message
    # and inject them into my_llm's system prompt so it can use them.
    memories = store.search(
        ("memories",),
        query=messages[-1]["content"],
    )

    response = my_llm.invoke(
        [
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant.\n\n"
                    "Use the following memories to personalize your responses:\n"
                    f"<memories>\n{memories}\n</memories>"
                ),
            },
            *messages,
        ]
    )

    # Step 2: After responding, let the manager agent extract and store new facts.
    # This runs after the user already has their answer (could also run in background).
    manager.invoke({"messages": messages})

    return response


/var/folders/j2/7cy_4n7j38nc1x8_k8zc9tbh0000gn/T/ipykernel_29914/3135083221.py:48: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  manager = create_react_agent(


## Demo: Memory Across Multiple Turns

Each `app.invoke()` call is a separate conversation turn (simulating a new session).  
After each turn, we print what's in the store so you can see memory accumulate.

In [2]:
# ── TURN 1: Introduce facts ────────────────────────────────────────────────────
# Store is empty — my_llm answers from the message alone.
# After responding, the manager agent extracts and stores what it learned.

print("=" * 60)
print("TURN 1")
print("=" * 60)

response1 = app.invoke(
    [{"role": "user", "content": "I'm Karthick. I love Python and I work on ML systems."}]
)
print("Assistant:", response1.content)

print("\n--- Store after Turn 1 ---")
for item in store.search(("memories",)):
    print(" •", item.value["content"])


TURN 1
Assistant: Hey Karthick! Great to meet you! 👋

That's a solid combination - Python and ML systems. You're essentially working with one of the most powerful tech stacks right now.

A few things I'd love to know more about:
- **What kind of ML systems** do you work on? (NLP, computer vision, recommendation systems, etc.)
- **What's your stack** like? (PyTorch, TensorFlow, scikit-learn, MLflow, etc.)
- **Any interesting challenges** you're tackling currently?

Feel free to dive into anything - whether it's debugging, architecture decisions, optimizations, or just talking shop about Python and ML! 🐍🤖

--- Store after Turn 1 ---
 • User's name is Karthick. He loves Python and works on ML (Machine Learning) systems.


In [3]:
# ── TURN 2: Ask something that requires memory ─────────────────────────────────
# The store now has facts from Turn 1.
# my_llm will search the store, find "Karthick loves Python / works on ML",
# and use that to give a personalised answer — even though this is a fresh message.

print("=" * 60)
print("TURN 2")
print("=" * 60)

response2 = app.invoke(
    [{"role": "user", "content": "What programming language should I learn next?"}]
)
print("Assistant:", response2.content)

print("\n--- Store after Turn 2 ---")
for item in store.search(("memories",)):
    print(" •", item.value["content"])


TURN 2
Assistant: Great question, Karthick! Since you already love Python and work on ML systems, here are some great options depending on your goals:

## Top Recommendations for You

### 1. **Rust** 🦀
- Excellent for performance-critical ML infrastructure
- Great for building fast, safe systems-level code
- Increasingly popular in the ML tooling ecosystem (e.g., Hugging Face's `tokenizers` library)

### 2. **C++** ⚡
- Essential for deep ML framework development (TensorFlow, PyTorch internals)
- Great for optimizing inference engines and embedded ML
- High performance for production ML systems

### 3. **Julia** 📊
- Designed for scientific computing and numerical analysis
- Syntax feels familiar to Python users
- Great for research-heavy ML work

### 4. **Go (Golang)** 🐹
- Excellent for building scalable ML serving infrastructure and APIs
- Simple, fast, and great for microservices
- Good complement to your Python ML work

### 5. **SQL (Advanced)** 🗄️
- Often overlooked but critical for

In [4]:
# ── TURN 3: Update a fact — manager should overwrite stale memory ──────────────
# The manager agent sees "Karthick now works on NLP" alongside the existing
# "works on ML systems" memory, and should update or extend it.

print("=" * 60)
print("TURN 3")
print("=" * 60)

response3 = app.invoke(
    [{"role": "user", "content": "I switched teams — I now work on NLP, not ML systems anymore."}]
)
print("Assistant:", response3.content)

print("\n--- Store after Turn 3 ---")
for item in store.search(("memories",)):
    print(" •", item.value["content"])


TURN 3
Assistant: Thanks for the update, Karthick! I'll remember that you're now working on **NLP** instead of ML systems. That's an exciting area — lots of interesting challenges around language models, text processing, and more. Are you working on anything specific in NLP that you'd like to discuss?

--- Store after Turn 3 ---
 • User's name is Karthick. He loves Python and works on NLP (Natural Language Processing) systems. He previously worked on ML (Machine Learning) systems.


In [5]:
# ── TURN 4: Verify memory is being used ───────────────────────────────────────
# Ask a question that should be answered using the updated memory from Turn 3.
# A good response here proves my_llm is reading the store — not just the message.

print("=" * 60)
print("TURN 4")
print("=" * 60)

response4 = app.invoke(
    [{"role": "user", "content": "What do you know about me?"}]
)
print("Assistant:", response4.content)


TURN 4
Assistant: Based on my memories, here's what I know about you:

- **Your name** is **Karthick**
- **You love Python** 🐍
- **You work on NLP (Natural Language Processing) systems**
- **You previously worked on ML (Machine Learning) systems**

Is there anything else you'd like to share or update about yourself? 😊
